# 02 — Spark SQL lineage with Spline

Same NYC Yellow Taxi sample, queries expressed in Spark SQL.
Each persisted `CREATE TABLE AS` produces a Spline lineage event.
Transformations are inlined in the write — not split across temp views.

Inspect results at <http://localhost:9090>.

## Lineage cheat sheet

| Your transformation | Spline node to click | Column to inspect |
|--------------------|----------------------|-------------------|
| `UNIX_TIMESTAMP` diff | `Project` | `trip_minutes` |
| `GROUP BY` + `SUM` | `Aggregate` | `revenue`, `trips` |
| Column select | `Project` | dropped columns absent in output |

In Spline UI: Execution Plan → turn off Compact view → click the node →
Output Schema → select a column → Lineage.


In [ ]:
from _shared.spark_session import get_spark, SAMPLE_CSV, PARQUET_SINK

spark = get_spark()
spark.sparkContext.setLogLevel('WARN')

spark.read.csv(SAMPLE_CSV, header=True, inferSchema=True).createOrReplaceTempView('taxi_raw')
print('raw row count:', spark.sql('SELECT COUNT(*) FROM taxi_raw').first()[0])


## Persist lineage (full SQL in each write)

Both statements contain the full transformation and write Parquet tables.
Spline should show `Project` (trip duration calc) and `Aggregate`
(zone revenue) directly on these execution plans.


In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE local.taxi.trip_durations_sql USING parquet AS
    SELECT
        PULocationID,
        DOLocationID,
        payment_type,
        trip_distance,
        (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60.0 AS trip_minutes,
        fare_amount,
        tip_amount,
        total_amount
    FROM taxi_raw
""")

spark.sql("""
    CREATE OR REPLACE TABLE local.taxi.zone_revenue_sql USING parquet AS
    SELECT PULocationID,
           COUNT(*) AS trips,
           ROUND(SUM(fare_amount), 2) AS revenue,
           ROUND(AVG(tip_amount), 2) AS avg_tip
    FROM taxi_raw
    GROUP BY PULocationID
    ORDER BY revenue DESC
""")

print('wrote trip_durations_sql and zone_revenue_sql (parquet)')


## Inspect lineage (Consumer API)

Text fallback when the graph UI feels opaque. Also open
<http://localhost:9090> and follow the cheat sheet above.

Consumer API docs: <http://localhost:8080/docs/consumer.html>


In [ ]:
from _shared.lineage_inspect import list_recent_events, inspect_event_column

list_recent_events()
inspect_event_column('trip_durations_sql', 'trip_minutes')
inspect_event_column('zone_revenue_sql', 'revenue')
